In [1]:
import os
import yfinance as yf
import pandas as pd
from sqlalchemy import create_engine

In [ ]:
import yfinance as yf

df = yf.download("AAPL", start="2020-01-01", end="2020-12-31")
df.head()

/tmp/ipykernel_2337/3434155883.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("AAPL", start="2020-01-01", end="2020-12-31")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-01-02,72.468277,72.528597,71.223274,71.476615,135480400
2020-01-03,71.763718,72.523746,71.539330,71.696160,146322800
2020-01-06,72.335556,72.374162,70.634539,70.885472,118387200
2020-01-07,71.995346,72.600952,71.775781,72.345197,108872000
2020-01-08,73.153496,73.455095,71.698581,71.698581,132079200


In [3]:
df = yf.download("NKE", start="2020-01-01", end="2020-12-31")
df.head()

/tmp/ipykernel_2337/898432130.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("NKE", start="2020-01-01", end="2020-12-31")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,NKE,NKE,NKE,NKE,NKE
Date,,,,,
2020-01-02,94.875160,94.884445,93.779732,94.095368,5644100
2020-01-03,94.615211,94.689479,93.120602,93.380533,4541800
2020-01-06,94.531677,94.540955,93.640483,93.714744,4612400
2020-01-07,94.485245,95.376439,93.584765,94.513093,6719900
2020-01-08,94.271759,94.819469,93.621925,94.048955,4942200


In [4]:

df = yf.download("MBG.DE", start="2020-01-01", end="2020-12-31")
df.head()

/tmp/ipykernel_2337/488092430.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("MBG.DE", start="2020-01-01", end="2020-12-31")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MBG.DE,MBG.DE,MBG.DE,MBG.DE,MBG.DE
Date,,,,,
2020-01-02,28.898632,29.092351,28.485177,28.569024,2973654
2020-01-03,28.375305,28.756958,28.019675,28.710698,4014262
2020-01-06,28.222065,28.222065,27.603326,27.950284,3441396
2020-01-07,28.447592,28.649983,28.282786,28.363745,2816618
2020-01-08,28.606609,28.621066,28.149784,28.219174,2659070


In [5]:
import os

TICKERS = os.getenv("TICKERS", "AAPL,MSFT,SPY").split(",")
TICKERS

['AAPL', 'NKE', 'MBG.DE']

In [6]:

# Leer variables de entorno
PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB = os.getenv("PG_DB", "trading_db")
PG_USER = os.getenv("PG_USER", "trading_user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "trading_pass")

print(PG_HOST, PG_DB, PG_USER)  # opcional, solo para verificar

# Crear engine
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

postgres trading_db trading_user


In [7]:
# Celda 1: Configuración e infraestructura

import os
import pandas as pd
import yfinance as yf

from sqlalchemy import create_engine, text

# --- Leer parámetros desde variables de entorno ---
PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB = os.getenv("PG_DB", "trading_db")
PG_USER = os.getenv("PG_USER", "trading_user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "trading_pass")
PG_SCHEMA_RAW = os.getenv("PG_SCHEMA_RAW", "raw")

TICKERS = os.getenv("TICKERS", "AAPL,NKE,MBG.DE").split(",")
START_DATE = os.getenv("START_DATE", "2021-01-01")
END_DATE = os.getenv("END_DATE", "2025-12-31")
RUN_ID = os.getenv("RUN_ID", "run_001")
DATA_PROVIDER = os.getenv("DATA_PROVIDER", "yfinance")

print("Postgres:", PG_HOST, PG_PORT, PG_DB, PG_USER)
print("Schema RAW:", PG_SCHEMA_RAW)
print("Tickers:", TICKERS)
print("Rango fechas:", START_DATE, "→", END_DATE)
print("RUN_ID:", RUN_ID)
print("DATA_PROVIDER:", DATA_PROVIDER)

# --- Crear engine de conexión a Postgres ---
engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

# Probar conexión rápida
with engine.connect() as conn:
    conn.execute(text("SELECT 1;"))
print("Conexión a Postgres OK ✅")

Postgres: postgres 5432 trading_db trading_user
Schema RAW: raw
Tickers: ['AAPL', 'NKE', 'MBG.DE']
Rango fechas: 2018-01-01 → 2025-12-31
RUN_ID: run_001
DATA_PROVIDER: yfinance
Conexión a Postgres OK ✅


In [8]:
# Celda 2: Definir esquema de raw.prices_daily desde cero

ddl_schema = """
CREATE SCHEMA IF NOT EXISTS raw;
"""

ddl_drop = """
DROP TABLE IF EXISTS raw.prices_daily;
"""

ddl_create = """
CREATE TABLE raw.prices_daily (
    date            DATE            NOT NULL,
    ticker          TEXT            NOT NULL,
    open            DOUBLE PRECISION,
    high            DOUBLE PRECISION,
    low             DOUBLE PRECISION,
    close           DOUBLE PRECISION,
    adj_close       DOUBLE PRECISION,
    volume          BIGINT,
    run_id          TEXT            NOT NULL,
    ingested_at_utc TIMESTAMPTZ     NOT NULL DEFAULT (NOW() AT TIME ZONE 'UTC'),
    source_name     TEXT            NOT NULL,
    CONSTRAINT pk_prices_daily PRIMARY KEY (date, ticker, run_id)
);
"""

ddl_index = """
CREATE INDEX IF NOT EXISTS idx_prices_daily_ticker_date
    ON raw.prices_daily (ticker, date);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_schema))
    conn.execute(text(ddl_drop))
    conn.execute(text(ddl_create))
    conn.execute(text(ddl_index))

print("Tabla raw.prices_daily recreada desde cero ✅")

Tabla raw.prices_daily recreada desde cero ✅


In [9]:
# Celda 3: Función de descarga desde Yahoo Finance

def descargar_precios_yahoo(ticker, start_date, end_date):
    """
    Descarga precios diarios OHLCV desde Yahoo Finance para un ticker.
    Devuelve un DataFrame con columnas estándar (no MultiIndex).
    """
    print(f"\nDescargando datos para {ticker} ...")
    df = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        progress=False,
        auto_adjust=False,   # queremos adj_close separado
        group_by="column"    # 🔑 fuerza columnas simples
    )

    if df.empty:
        print(f"⚠️  Sin datos para {ticker} en el rango {start_date}–{end_date}")
        return pd.DataFrame()

    # Si por alguna razón aún viene MultiIndex, lo aplanamos usando el primer nivel
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]

    # Renombrar columnas al estándar definido
    df = df.rename(
        columns={
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
        }
    )

    # Resetear índice (Date) a columna normal
    df = df.reset_index()  # "Date" pasa a columna

    # Añadir columnas obligatorias
    df["ticker"] = ticker
    df["run_id"] = RUN_ID
    df["source_name"] = DATA_PROVIDER
    df["ingested_at_utc"] = pd.Timestamp.utcnow()

    # Dejar solo columnas en el orden esperado por raw.prices_daily
    df = df[
        [
            "Date",
            "ticker",
            "open",
            "high",
            "low",
            "close",
            "adj_close",
            "volume",
            "run_id",
            "ingested_at_utc",
            "source_name",
        ]
    ]

    df = df.rename(columns={"Date": "date"})
    df["date"] = pd.to_datetime(df["date"]).dt.date

    print(
        f"   -> {len(df)} filas descargadas para {ticker}. "
        f"Rango: {df['date'].min()} → {df['date'].max()}"
    )
    return df

In [10]:
# Celda 4: Ingesta a raw.prices_daily con idempotencia y resumen

all_rows = []

for ticker in TICKERS:
    df_ticker = descargar_precios_yahoo(
        ticker=ticker,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    if not df_ticker.empty:
        all_rows.append(df_ticker)

if not all_rows:
    raise ValueError("No se descargó información para ningún ticker. Revisa parámetros.")

prices_df = pd.concat(all_rows, ignore_index=True)
print(f"\nTotal filas descargadas (todos los tickers): {len(prices_df)}")

# --- Idempotencia: borrar previamente el rango para estos tickers ---
min_date = prices_df["date"].min()
max_date = prices_df["date"].max()
tickers_param = prices_df["ticker"].unique().tolist()

print(
    f"\nAplicando idempotencia: eliminando filas previas "
    f"de raw.prices_daily para tickers={tickers_param}, "
    f"rango={min_date} → {max_date}"
)

delete_sql = text("""
    DELETE FROM raw.prices_daily
    WHERE date BETWEEN :start_date AND :end_date
      AND ticker = :ticker
""")

with engine.begin() as conn:
    for t in tickers_param:
        conn.execute(
            delete_sql,
            {
                "start_date": min_date,
                "end_date": max_date,
                "ticker": t,
            },
        )

print("Filas previas eliminadas (si existían). Insertando nuevas filas...")

# --- Insertar en raw.prices_daily ---
prices_df.to_sql(
    "prices_daily",
    con=engine,
    schema=PG_SCHEMA_RAW,
    if_exists="append",
    index=False,
)

print("Inserción completada en raw.prices_daily ✅")


Descargando datos para AAPL ...
   -> 1988 filas descargadas para AAPL. Rango: 2018-01-02 → 2025-11-26

Descargando datos para NKE ...
   -> 1988 filas descargadas para NKE. Rango: 2018-01-02 → 2025-11-26

Descargando datos para MBG.DE ...
   -> 2011 filas descargadas para MBG.DE. Rango: 2018-01-02 → 2025-11-27

Total filas descargadas (todos los tickers): 5987

Aplicando idempotencia: eliminando filas previas de raw.prices_daily para tickers=['AAPL', 'NKE', 'MBG.DE'], rango=2018-01-02 → 2025-11-27
Filas previas eliminadas (si existían). Insertando nuevas filas...
Inserción completada en raw.prices_daily ✅


In [11]:
# Celda 5: Resumen de validación de la ingesta

query_resumen = """
    SELECT
        ticker,
        COUNT(*) AS n_rows,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM raw.prices_daily
    GROUP BY ticker
    ORDER BY ticker;
"""

resumen_df = pd.read_sql(query_resumen, engine)
resumen_df

,ticker,n_rows,min_date,max_date
0,AAPL,1988,2018-01-02,2025-11-26
1,MBG.DE,2011,2018-01-02,2025-11-27
2,NKE,1988,2018-01-02,2025-11-26


In [12]:
from sqlalchemy import text

ddl_schema_analytics = """
CREATE SCHEMA IF NOT EXISTS analytics;
"""

ddl_drop_daily_features = """
DROP TABLE IF EXISTS analytics.daily_features;
"""

ddl_create_daily_features = """
CREATE TABLE analytics.daily_features (
    date                DATE            NOT NULL,
    ticker              TEXT            NOT NULL,
    year                INTEGER         NOT NULL,
    month               INTEGER         NOT NULL,
    day_of_week         INTEGER         NOT NULL,   -- 0=lunes ... 6=domingo (por ejemplo)

    open                DOUBLE PRECISION,
    close               DOUBLE PRECISION,
    high                DOUBLE PRECISION,
    low                 DOUBLE PRECISION,
    volume              BIGINT,

    return_close_open   DOUBLE PRECISION,           -- (close - open) / open
    return_prev_close   DOUBLE PRECISION,           -- close / close_lag1 - 1
    volatility_n_days   DOUBLE PRECISION,           -- std de retornos últimos N días (ej: 10)

    is_monday           BOOLEAN,
    is_friday           BOOLEAN,

    run_id              TEXT            NOT NULL,
    ingested_at_utc     TIMESTAMPTZ     NOT NULL DEFAULT (NOW() AT TIME ZONE 'UTC'),

    CONSTRAINT pk_daily_features PRIMARY KEY (date, ticker)
);
"""

ddl_index_daily_features = """
CREATE INDEX IF NOT EXISTS idx_daily_features_ticker_date
    ON analytics.daily_features (ticker, date);
"""

with engine.begin() as conn:
    conn.execute(text(ddl_schema_analytics))
    conn.execute(text(ddl_drop_daily_features))
    conn.execute(text(ddl_create_daily_features))
    conn.execute(text(ddl_index_daily_features))

print("Tabla analytics.daily_features recreada desde cero ✅")

Tabla analytics.daily_features recreada desde cero ✅


In [13]:
df_feat_summary = pd.read_sql("""
    SELECT
        ticker,
        COUNT(*) AS n_rows,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM analytics.daily_features
    GROUP BY ticker
    ORDER BY ticker;
""", engine)

df_feat_summary

PendingRollbackError: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)

In [14]:
df_sample = pd.read_sql("""
    SELECT *
    FROM analytics.daily_features
    WHERE ticker = 'AAPL'
    ORDER BY date
    LIMIT 10;
""", engine)

df_sample

,date,ticker,year,month,day_of_week,open,close,high,low,volume,return_close_open,return_prev_close,volatility_n_days,is_monday,is_friday,run_id,ingested_at_utc
0,2018-01-02,AAPL,2018,1,1,42.540001,43.064999,43.075001,42.314999,102223600,0.012341,NaN,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
1,2018-01-03,AAPL,2018,1,2,43.132500,43.057499,43.637501,42.990002,118071600,-0.001739,-0.000174,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
2,2018-01-04,AAPL,2018,1,3,43.134998,43.257500,43.367500,43.020000,89738400,0.002840,0.004645,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
3,2018-01-05,AAPL,2018,1,4,43.360001,43.750000,43.842499,43.262501,94640000,0.008994,0.011385,None,False,True,feat_run_001,2025-11-27 17:55:05.477696+00:00
4,2018-01-08,AAPL,2018,1,0,43.587502,43.587502,43.902500,43.482498,82271200,0.000000,-0.003714,None,True,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
5,2018-01-09,AAPL,2018,1,1,43.637501,43.582500,43.764999,43.352501,86336000,-0.001260,-0.000115,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
6,2018-01-10,AAPL,2018,1,2,43.290001,43.572498,43.575001,43.250000,95839600,0.006526,-0.000229,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
7,2018-01-11,AAPL,2018,1,3,43.647499,43.820000,43.872501,43.622501,74670800,0.003952,0.005680,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00
8,2018-01-12,AAPL,2018,1,4,44.044998,44.272499,44.340000,43.912498,101672400,0.005165,0.010326,None,False,True,feat_run_001,2025-11-27 17:55:05.477696+00:00
9,2018-01-16,AAPL,2018,1,1,44.474998,44.047501,44.847500,44.035000,118263600,-0.009612,-0.005082,None,False,False,feat_run_001,2025-11-27 17:55:05.477696+00:00


In [15]:
from sqlalchemy import create_engine

engine.dispose()  # si existe

PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB = os.getenv("PG_DB", "trading_db")
PG_USER = os.getenv("PG_USER", "trading_user")
PG_PASSWORD = os.getenv("PG_PASSWORD", "trading_pass")

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

In [16]:
df_feat_summary = pd.read_sql("""
    SELECT
        ticker,
        COUNT(*) AS n_rows,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM analytics.daily_features
    GROUP BY ticker
    ORDER BY ticker;
""", engine)

df_feat_summary

,ticker,n_rows,min_date,max_date
0,AAPL,1988,2018-01-02,2025-11-26
1,MBG.DE,2011,2018-01-02,2025-11-27
2,NKE,1988,2018-01-02,2025-11-26
